# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GauriKale1328/flyrank-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

I will prioritize content pages for refresh when they show signs of opportunity:
high search visibility but weaker performance.

The baseline score will increase when:
- A page has high impressions, showing that users are searching for the topic.
- A page has low CTR compared to other pages, showing that the page may not attract enough clicks.
- A page has a weaker average search position, showing that improvement may be possible.

Pages with higher scores will appear higher in the refresh priority queue.

### Reason codes

The rule can output these reason codes:

1. HIGH_IMPRESSIONS_LOW_CTR
   - The page receives search visibility but fewer clicks.

2. POOR_SEARCH_POSITION
   - The page ranks lower and may benefit from optimization.

3. LOW_ENGAGEMENT
   - The page gets traffic but user engagement signals are weak.

4. MULTIPLE_OPPORTUNITY_SIGNALS
   - The page matches multiple refresh indicators.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb

con = duckdb.connect()


In [23]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

In [24]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN is not None)

True


In [25]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("HF login successful")

HF login successful


In [26]:
import duckdb

con = duckdb.connect()

print("DuckDB connected")

DuckDB connected


In [27]:
con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("httpfs loaded")

httpfs loaded


In [28]:
con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("HF secret created")

HF secret created


In [29]:
df = con.execute("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 100000
""").df()

df.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(100000, 31)

In [30]:
df["baseline_score"] = 0

# High impressions = more opportunity
high_impressions = df["gsc_impressions"] > df["gsc_impressions"].quantile(0.75)

# Low CTR = many impressions but fewer clicks
low_ctr = (
    df["gsc_clicks"] / (df["gsc_impressions"] + 1)
) < 0.02

# Poor ranking position
poor_position = df["gsc_sum_position"] > 20


df.loc[high_impressions, "baseline_score"] += 2
df.loc[low_ctr, "baseline_score"] += 2
df.loc[poor_position, "baseline_score"] += 1


df[["baseline_score"]].head()

,baseline_score
0,5
1,2
2,5
3,3
4,3


In [31]:
def create_reason(row):
    reasons = []

    ctr = row["gsc_clicks"] / (row["gsc_impressions"] + 1)

    if row["gsc_impressions"] > df["gsc_impressions"].quantile(0.75):
        reasons.append("HIGH_IMPRESSIONS")

    if ctr < 0.02:
        reasons.append("LOW_CTR")

    if row["gsc_sum_position"] > 20:
        reasons.append("POOR_POSITION")

    if len(reasons) == 0:
        return "NO_SIGNAL"

    return "_".join(reasons)


df["reason_code"] = df.apply(create_reason, axis=1)

df[["baseline_score", "reason_code"]].head()

,baseline_score,reason_code
0,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION
1,2,LOW_CTR
2,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION
3,3,LOW_CTR_POOR_POSITION
4,3,LOW_CTR_POOR_POSITION


In [32]:
df["action"] = df["baseline_score"].apply(
    lambda x: "REFRESH_RECOMMENDED" if x >= 3 else "MONITOR"
)

df[["baseline_score", "reason_code", "action"]].head()

,baseline_score,reason_code,action
0,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
1,2,LOW_CTR,MONITOR
2,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
3,3,LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
4,3,LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED


In [33]:
import os

os.makedirs("work/outputs", exist_ok=True)

print("Output folder ready")

Output folder ready


In [34]:
# Rank pages by baseline score

queue = df.sort_values(
    by="baseline_score",
    ascending=False
)

# Select required columns
baseline_queue = queue[
    [
        "content_hash_id",
        "client_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

# Write CSV
baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

baseline_queue.head(10)

,content_hash_id,client_hash_id,baseline_score,reason_code,action
83437,content_d524f13ae7418275,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83438,content_ded5dd5859f8be00,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83443,content_be2da737f8da4b33,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
33025,content_39847134f0c27d32,client_e547b89c05043229,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83447,content_acf7716b5104cf44,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83448,content_98648444f3cc6e80,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
33024,content_7ea4bc1946d7cff3,client_e547b89c05043229,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83452,content_60a6eae999c294cb,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83453,content_946c37b5b6603a0b,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83454,content_11b595040966802b,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

### 1. content_hash_id_here

- Action: REFRESH_RECOMMENDED
- Reason code: HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION
- Confidence note: Medium confidence because the page has multiple opportunity signals, including visibility and weak click performance.
- What would make it wrong: The low CTR may be caused by search intent changes or SERP competition rather than outdated content.

### 2. content_hash_id_here

- Action: REFRESH_RECOMMENDED
- Reason code: LOW_CTR_POOR_POSITION
- Confidence note: Medium confidence because ranking and click signals indicate possible improvement opportunity.
- What would make it wrong: The page may already satisfy users despite weaker search metrics.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = baseline_queue.head(20)

top20


,content_hash_id,client_hash_id,baseline_score,reason_code,action
83437,content_d524f13ae7418275,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83438,content_ded5dd5859f8be00,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83443,content_be2da737f8da4b33,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
33025,content_39847134f0c27d32,client_e547b89c05043229,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83447,content_acf7716b5104cf44,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83448,content_98648444f3cc6e80,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
33024,content_7ea4bc1946d7cff3,client_e547b89c05043229,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83452,content_60a6eae999c294cb,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83453,content_946c37b5b6603a0b,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED
83454,content_11b595040966802b,client_c182d11e4862a37d,5,HIGH_IMPRESSIONS_LOW_CTR_POOR_POSITION,REFRESH_RECOMMENDED


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Some pages selected by the baseline rule may not actually need a refresh.

- A page with high impressions and low CTR may be affected by changing search intent, strong SERP competition, or title/metadata issues instead of outdated content.
- A page with poor average position may be difficult to improve because of external ranking factors or competitive topics.
- Low engagement signals may not always indicate poor content quality because user behavior can vary by content type.

These cases show that the baseline score is a prioritization tool, not a final decision.

### Leakage check

I confirmed that the baseline rule only uses information available at the decision time.

- No future performance windows were used.
- No label-derived columns were used.
- No product flags or existing outcome flags were used as features.
- Identifiers are used only for organizing the ranked queue, not for prediction.

The baseline uses observed search and engagement signals available in the selected month to create a decision-support ranking.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.